# 3. Subtract the channel model and smooth

Use aperture spectra from the smoothed VLA cube to distribute each source's integrated sidelobe map across the FAST channels. Subtract the summed model and apply the original smoothing kernel.

In [ ]:
import os
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from spectral_cube import SpectralCube
from photutils.aperture import CircularAperture, aperture_photometry
import matplotlib.pyplot as plt

# -----------------------------
# Input / output
# -----------------------------
fast_cube_file = './rawcube/HCG100-2fov-HIcube-pix1.4arcmin-20kms-1-mJyBeam-1.fits'
side_m0_dir = './sidelobe-result-fits/VLAgrid_method_FASTgrid_M0'
vla_smooth_cube_file = './VLA/vla_smooth_3arcmin_on_fastgrid_5000_5800_mJyBeam.fits'

outdir = './sidelobe-result-fits/VLAgrid_method_channel_subtraction'
os.makedirs(outdir, exist_ok=True)

# -----------------------------
# Read FAST raw cube
# -----------------------------
u.add_enabled_units(u.def_unit(['mJy/Beam'], represents=u.Jy/u.beam/1000))
fast_cube = SpectralCube.read(fast_cube_file)

cube = fast_cube.filled_data[:].value
header = fast_cube.header.copy()
vel = fast_cube.spectral_axis.to(u.km/u.s).value

nchan, ny, nx = cube.shape
dv = np.nanmedian(np.abs(np.diff(vel)))

print('cube shape:', cube.shape)
print('velocity range:', vel[0], vel[-1])
print('dv =', dv)

fast_wcs_2d = WCS(header).celestial

# Read smoothed VLA cube on FAST grid
vla_smooth_cube = SpectralCube.read(vla_smooth_cube_file)
nchan_vla, ny_vla, nx_vla = vla_smooth_cube.shape

fast_vel = vel          # 140 channels, 3960–6740
vla_vel = vla_smooth_cube.spectral_axis.to(u.km/u.s).value  # 41 channels, 5000–5800
vla_smooth_data = vla_smooth_cube.filled_data[:].value       # shape (41,34,28)
vla_smooth_data = np.nan_to_num(vla_smooth_data, nan=0.0)

# 如果 VLA smoothed cube 速度轴和 FAST raw cube 已经一致，可以直接用
# 否则需要插值到 FAST velocity axis
print(vla_smooth_data.shape, cube.shape)
print(vla_vel[:5], vel[:5])

In [ ]:
sources = {
    'HCG100': {
        'ra': 0.3022699,
        'dec': 13.0891334,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 4.0,
        'side_m0_file': f'{side_m0_dir}/sidelobe_HCG100_M0_FASTgrid.fits',
    },
    'NGC7810': {
        'ra': 0.5763757,
        'dec': 12.9642823,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 2.2,
        'side_m0_file': f'{side_m0_dir}/sidelobe_NGC7810_M0_FASTgrid.fits',
    },
    'MRK935': {
        'ra': 0.4504456,
        'dec': 13.1028654,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 1.75,
        'side_m0_file': f'{side_m0_dir}/sidelobe_MRK935_M0_FASTgrid.fits',
    },
    'AGC105092': {
        'ra': 0.4095431,
        'dec': 12.9043733,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 2.2,
        'side_m0_file': f'{side_m0_dir}/sidelobe_AGC105092_M0_FASTgrid.fits',
    },
}

def arcmin_to_pixel_radius(wcs2d, radius_arcmin):
    pixscale = u.Quantity(wcs2d.proj_plane_pixel_scales(), u.deg)
    pix_x = abs(pixscale[0].to(u.arcmin).value)
    pix_y = abs(pixscale[1].to(u.arcmin).value)
    return radius_arcmin / np.sqrt(pix_x * pix_y)

for name, src in sources.items():
    coord = SkyCoord(src['ra']*u.deg, src['dec']*u.deg)
    x, y = fast_wcs_2d.world_to_pixel(coord)
    r_pix = arcmin_to_pixel_radius(fast_wcs_2d, src['radius_arcmin'])

    src['xpix'] = x
    src['ypix'] = y
    src['rpix'] = r_pix
    src['aperture'] = CircularAperture((x, y), r=r_pix)

    print(name, x, y, r_pix)

In [ ]:
def make_channel_scale_from_vla_smooth(
    vla_data,
    vla_vel,
    fast_vel,
    aperture,
    vmin,
    vmax,
    dv_fast=20.0
):
    """
    Extract source spectrum from smoothed VLA cube and return
    channel_scale on the full FAST velocity axis.

    vla_data: shape = (nchan_vla, ny, nx)
    vla_vel:  velocity axis of VLA smoothed cube
    fast_vel: full FAST velocity axis
    output: channel_scale_full, same length as fast_vel
    """

    # spectrum on VLA channels
    spec_vla = np.zeros(len(vla_vel), dtype=float)

    vmask_vla = (vla_vel >= vmin) & (vla_vel <= vmax)

    print('velocity range:', vmin, vmax)
    print('VLA channels in range:', np.sum(vmask_vla))
    print('VLA velocity covered:', np.nanmin(vla_vel), np.nanmax(vla_vel))

    for i in range(len(vla_vel)):
        if vmask_vla[i]:
            phot_table = aperture_photometry(vla_data[i, :, :], aperture)
            spec_vla[i] = phot_table['aperture_sum'][0]
        else:
            spec_vla[i] = 0.0

    # clean negative values only for normalization
    spec_clean = spec_vla.copy()
    spec_clean[spec_clean < 0] = 0.0

    norm = np.nansum(spec_clean) * dv_fast

    print('spec_vla min/max:', np.nanmin(spec_vla), np.nanmax(spec_vla))
    print('spec_clean sum:', np.nansum(spec_clean))
    print('norm:', norm)

    if norm <= 0:
        raise ValueError('Bad spectral normalization. Check velocity range, aperture, or VLA cube values.')

    channel_scale_vla = spec_clean / norm
    # unit: 1 / km/s

    # Put it onto full FAST velocity axis
    channel_scale_full = np.zeros(len(fast_vel), dtype=float)

    for i_fast, vf in enumerate(fast_vel):
        # exact/nearest matching
        j = np.argmin(np.abs(vla_vel - vf))

        if np.abs(vla_vel[j] - vf) < 0.5 * dv_fast:
            channel_scale_full[i_fast] = channel_scale_vla[j]
        else:
            channel_scale_full[i_fast] = 0.0

    print('check sum full scale * dv =', np.nansum(channel_scale_full) * dv_fast)

    return channel_scale_full, spec_vla, spec_clean

In [ ]:
# for name, src in sources.items():

#     vmask = (vel >= src['vmin']) & (vel <= src['vmax'])

#     spec = np.zeros(nchan, dtype=float)

#     for i in range(nchan):
#         if vmask[i]:
#             phot_table = aperture_photometry(cube[i, :, :], src['aperture'])
#             spec[i] = phot_table['aperture_sum'][0]
#         else:
#             spec[i] = 0.0

#     # 用于归一化的谱。可以把负值置零，避免噪声导致异常归一化。
#     plt.step(vel, spec)
#     spec_clean = spec.copy()
#     spec_clean[spec_clean < 0] = 0.0

#     norm = np.nansum(spec_clean) * dv

#     if norm <= 0:
#         raise ValueError(f'Bad spectral normalization for {name}')

#     channel_scale = spec_clean / norm
#     # unit: 1 / (km/s)

#     src['spec'] = spec
#     src['spec_clean'] = spec_clean
#     src['channel_scale'] = channel_scale

#     print(name)
#     print('  norm =', norm)
#     print('  sum(channel_scale * dv) =', np.nansum(channel_scale) * dv)
#     print('  max channel_scale =', np.nanmax(channel_scale))

In [ ]:
for name, src in sources.items():

    channel_scale_full, spec_vla, spec_clean = make_channel_scale_from_vla_smooth(
        vla_smooth_data,
        vla_vel,
        vel,                 # FAST full velocity axis
        src['aperture'],
        src['vmin'],
        src['vmax'],
        dv_fast=dv
    )

    src['channel_scale'] = channel_scale_full
    src['spec_vla'] = spec_vla
    src['spec_clean_vla'] = spec_clean

    print(name)
    print('sum(channel_scale * dv) =', np.nansum(channel_scale_full) * dv)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

for name, src in sources.items():
    plt.step(vla_vel, src['spec_clean_vla'], where='mid', label=name)

plt.xlim(5000, 5800)
plt.xlabel('Velocity [km/s]')
plt.ylabel('Aperture sum from smoothed VLA cube')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
sidelobe_total_cube = np.zeros_like(cube, dtype=float)

for name, src in sources.items():

    side_m0 = fits.getdata(src['side_m0_file']).astype(float)

    if side_m0.shape != (ny, nx):
        raise ValueError(
            f'{name} shape mismatch: side_m0 {side_m0.shape}, cube {(ny, nx)}'
        )

    channel_scale = src['channel_scale']

    # side_m0: mJy/beam km/s
    # channel_scale: 1/(km/s)
    # side_cube_src: mJy/beam
    side_cube_src = side_m0[None, :, :] * channel_scale[:, None, None]

    sidelobe_total_cube += side_cube_src

    out_src = f'{outdir}/sidelobe_{name}_cube_scaled_to_channels.fits'
    h_src = header.copy()
    h_src['BUNIT'] = 'mJy/beam'
    h_src['COMMENT'] = f'Source-specific sidelobe cube for {name}.'
    h_src['COMMENT'] = 'M0 sidelobe map scaled to channels using the smoothed VLA aperture spectrum.'

    fits.writeto(
        out_src,
        side_cube_src.astype('float32'),
        h_src,
        overwrite=True
    )

# subtract
cube_side_sub = cube - sidelobe_total_cube

h_total = header.copy()
h_total['BUNIT'] = 'mJy/beam'
h_total['COMMENT'] = 'Total sidelobe cube from VLA-grid beam-convolution method.'
h_total['COMMENT'] = 'Source-specific M0 sidelobe maps scaled to channels and summed.'

fits.writeto(
    f'{outdir}/sidelobe_total_cube_scaled_to_channels.fits',
    sidelobe_total_cube.astype('float32'),
    h_total,
    overwrite=True
)

h_corr = header.copy()
h_corr['BUNIT'] = 'mJy/beam'
h_corr['COMMENT'] = 'FAST raw cube after source-specific sidelobe subtraction.'
h_corr['COMMENT'] = 'Sidelobe maps derived from scaled VLA truth maps and FAST real-minus-Gaussian beams.'

fits.writeto(
    f'{outdir}/HCG100_FAST_cube_VLAgrid_sidelobe_sub.fits',
    cube_side_sub.astype('float32'),
    h_corr,
    overwrite=True
)

In [ ]:
from astropy.convolution import convolve, convolve_fft
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans

from scipy.ndimage import convolve as scipy_nd_convolve
#smooth to 4 arcmin


kernel = Gaussian2DKernel(x_stddev = 0.8344, y_stddev = 0.9631)
plt.imshow(kernel)

In [ ]:
print(np.array(kernel).sum())

plt.imshow(cube[75,:,:], origin='lower')

In [ ]:
image_cube_conv_astropy = cube_side_sub.copy()
image_cube_conv_scipy = cube_side_sub.copy()
for i in range(nchan):
    #print(i)
    image_cube_conv_astropy[i,:,:] = convolve(cube_side_sub[i,:,:], kernel, boundary='fill', fill_value=np.nan, mask = np.isnan(cube[i,:,:]), preserve_nan = True)



In [ ]:
i_check = 75

fig, axs = plt.subplots(1,2, figsize=(20,10))

ax = axs[0]
ax.imshow(cube[i_check,:,:], origin='lower', vmin = 0, vmax = 1)

ax = axs[1]
ax.imshow(image_cube_conv_astropy[i_check,:,:], origin='lower', vmin = 0, vmax = 1)



In [ ]:
smooth_header = header.copy()
smooth_header['BMAJ'] = 4./60. 
smooth_header['BMIN'] = 4./60. 

In [ ]:
fits.writeto('./HCG100-2fov-HIcube-pix1.4arcmin-20kms-1-mJyBeam-1-VLA-sidelobe-sub-smooth.fits', image_cube_conv_astropy, smooth_header, overwrite=True)
